Benchmark de Différents Modèles ML (94 Features, Sans Équilibrage)

In [1]:
# CELLULE 1 : IMPORTS ET CONFIGURATION
# =============================================================================
print("--- Cellule 1: Imports et Configuration ---")

import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC # Gardé pour référence, mais nous utiliserons LinearSVC
from sklearn.svm import LinearSVC # Importation pour l'optimisation
from sklearn.calibration import CalibratedClassifierCV # Pour les probabilités avec LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, average_precision_score
)

# LightGBM
try:
    import lightgbm as lgb
    lgbm_available = True
except ImportError:
    lgbm_available = False
    print("Warning: LightGBM non trouvé. Il sera ignoré.")

# XGBoost
try:
    import xgboost as xgb
    xgboost_available = True
except ImportError:
    xgboost_available = False
    print("Warning: XGBoost non trouvé. Il sera ignoré.")

# --- Configuration ---
DATA_PATH = "/home/kaoutar/Desktop/data/" # Modifiez si nécessaire
INPUT_FILE = os.path.join(DATA_PATH, 'dataset_X94_clean_enhanced.csv') # Modifiez si nécessaire
TARGET_COL_NAME = 'appetent'
RANDOM_STATE = 42
# Augmenter max_iter pour les solveurs itératifs si des problèmes de convergence apparaissent
LIBLINEAR_MAX_ITER = 2000 # Pour LogisticRegression et LinearSVC
MLP_MAX_ITER = 300
# ---------------------------------------------

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn') # Pour ignorer certains warnings de convergence non critiques

print("-" * 50)
print("Imports et Configuration Terminés.")
print(f"Fichier d'entrée : {INPUT_FILE}")
print("Stratégie d'équilibrage pour ce benchmark : AUCUNE (les modèles verront les données d'entraînement déséquilibrées).")
print("-" * 50)

--- Cellule 1: Imports et Configuration ---
--------------------------------------------------
Imports et Configuration Terminés.
Fichier d'entrée : /home/kaoutar/Desktop/data/dataset_X94_clean_enhanced.csv
Stratégie d'équilibrage pour ce benchmark : AUCUNE (les modèles verront les données d'entraînement déséquilibrées).
--------------------------------------------------


In [ ]:
# =============================================================================
# CELLULE 2 : CHARGEMENT DES DONNÉES (94 features) ET PRÉPARATION X, y
# =============================================================================
print(f"\n--- Chargement du jeu de données: {INPUT_FILE} ---")
X, y, data_loaded_successfully = None, None, False
if os.path.exists(INPUT_FILE):
    try:
        df_final_94_features = pd.read_csv(INPUT_FILE, sep=',')
        print(f"Données chargées. Shape initiale: {df_final_94_features.shape}")
        if TARGET_COL_NAME not in df_final_94_features.columns:
             raise ValueError(f"Colonne cible '{TARGET_COL_NAME}' manquante.")
        X = df_final_94_features.drop(TARGET_COL_NAME, axis=1).select_dtypes(include=[np.number]).fillna(0)
        y = df_final_94_features[TARGET_COL_NAME]
        if X.isna().sum().sum() > 0: X = X.fillna(0)
        if y.isna().any(): valid_indices = ~y.isna(); X = X.loc[valid_indices]; y = y.loc[valid_indices]
        if not pd.api.types.is_numeric_dtype(y): y = y.astype(int)
        unique_y_values = np.unique(y)
        if not set(unique_y_values).issubset({0, 1}): raise ValueError(f"Cible non binaire: {unique_y_values}")
        print(f"\nPréparation X/y OK. Shape X: {X.shape}, Shape y: {y.shape}")
        data_loaded_successfully = True
    except Exception as e: print(f"ERREUR chargement/préparation: {e}")
else: print(f"ERREUR: Fichier {INPUT_FILE} non trouvé.")
# Dans votre Cellule 2, après avoir chargé df_final_94_features
y = df_final_94_features[TARGET_COL_NAME]
X = df_final_94_features.drop(columns=[TARGET_COL_NAME, 'appetent']) # S'assurer que 'RADICAL' est le nom exact

# Vérifiez ensuite les shapes
print(f"Préparation X/y OK. Shape X: {X.shape}, Shape y: {y.shape}")
# X devrait maintenant avoir 93 colonnes si RADICAL était bien la 95ème colonne non-cible.
# Ou 94 si RADICAL n'était pas dans les colonnes de df_final_94_features utilisées pour X.
# Votre log actuel "Shape X: (1051281, 94)" est bon SI RADICAL n'est pas l'une de ces 94 features.
# --- FIN Cellule 2 ---


--- Chargement du jeu de données: /home/kaoutar/Desktop/data/dataset_X94_clean_enhanced.csv ---
Données chargées. Shape initiale: (1051281, 95)

Préparation X/y OK. Shape X: (1051281, 94), Shape y: (1051281,)


KeyError: "['RADICAL'] not found in axis"

In [3]:
# =============================================================================
# CELLULE 3 : PRÉTRAITEMENT (SPLIT ET SCALE - SANS RÉÉQUILIBRAGE DE DONNÉES)
# =============================================================================
X_train_scaled, X_test_scaled, y_train, y_test = None, None, None, None
data_ready_for_modeling = False
if data_loaded_successfully and X is not None and y is not None and not X.empty:
    print("\n--- Prétraitement ---")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    print(f"  Split OK. Train: {X_train.shape}, Test: {X_test.shape}")
    print("  Pas de rééchantillonnage appliqué. y_train conserve sa distribution originale.")
    y_train_final_for_model = y_train.copy()
    X_train_to_scale = X_train.copy()

    print("  Mise à l'échelle (StandardScaler)...")
    numerical_cols = X.select_dtypes(include=np.number).columns
    binary_cols = [col for col in X.select_dtypes(include='int').columns if X[col].nunique(dropna=False) <= 2]
    age_group_cols = [col for col in X.columns if col.startswith('age_group_')]
    cols_to_exclude_from_scaling = binary_cols + age_group_cols
    numerical_cols_to_scale = [c for c in X.columns if c in numerical_cols and c not in cols_to_exclude_from_scaling and c in X_train_to_scale.columns]

    scaler = StandardScaler()
    X_train_scaled = X_train_to_scale.copy()
    X_test_scaled = X_test.copy()
    if numerical_cols_to_scale:
         X_train_num_part = X_train_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_test_num_part = X_test_scaled[numerical_cols_to_scale].fillna(X_train[numerical_cols_to_scale].median())
         X_train_scaled.loc[:, numerical_cols_to_scale] = scaler.fit_transform(X_train_num_part)
         X_test_scaled.loc[:, numerical_cols_to_scale] = scaler.transform(X_test_num_part)
         print("  Scaling appliqué.")
    else: print("  Aucune colonne à scaler trouvée.")
    data_ready_for_modeling = True
else: print("Erreur: Données non prêtes.")
if data_ready_for_modeling:
    print(f"\n  Vérification des NaNs après prétraitement:")
    print(f"    NaNs dans X_train_scaled : {X_train_scaled.isna().sum().sum()}")
    print(f"    NaNs dans X_test_scaled  : {X_test_scaled.isna().sum().sum()}")
    # Si des NaNs sont présents, vous pourriez investiguer quelles colonnes sont concernées :
    # if X_train_scaled.isna().sum().sum() > 0:
    #     print("Colonnes avec NaNs dans X_train_scaled:")
    #     print(X_train_scaled.columns[X_train_scaled.isna().any()].tolist())
# --- FIN Cellule 3 ---


--- Prétraitement ---
  Split OK. Train: (841024, 94), Test: (210257, 94)
  Pas de rééchantillonnage appliqué. y_train conserve sa distribution originale.
  Mise à l'échelle (StandardScaler)...
  Scaling appliqué.


In [4]:
benchmark_results_all_models = {} # Initialiser ici pour collecter tous les résultats

if data_ready_for_modeling and X_train_scaled is not None and y_train_final_for_model is not None:
    print(f"\n--- Cellule 4A: Benchmark des Modèles ML (Sauf SVC) ---")
    print(f"--- (Entraînement sur Données Déséquilibrées) ---")

    models_to_benchmark_A = {
        "LogisticRegression": LogisticRegression(random_state=RANDOM_STATE, max_iter=LIBLINEAR_MAX_ITER, solver='liblinear', C=1), # Ajout de C=1 comme dans votre SVC
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=15, min_samples_leaf=5),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE, max_depth=5, learning_rate=0.1),
        "KNeighbors": KNeighborsClassifier(n_neighbors=11, n_jobs=-1),
        "MLPClassifier": MLPClassifier(random_state=RANDOM_STATE, max_iter=MLP_MAX_ITER, early_stopping=True, hidden_layer_sizes=(64,32))
    }
    if lgbm_available:
        models_to_benchmark_A["LightGBM"] = lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1)
    if xgboost_available:
        models_to_benchmark_A["XGBoost"] = xgb.XGBClassifier(objective='binary:logistic', random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')

    for name, model in models_to_benchmark_A.items():
        if model is None: continue
        print(f"\n--- Modèle Testé : {name} ---")
        try:
            start_time = datetime.now()
            print(f"  Entraînement {name}...")
            # S'assurer qu'il n'y a pas de NaN avant l'entraînement, même si le scaling devrait les avoir gérés
            current_X_train = X_train_scaled.fillna(0) # Fallback fillna
            model.fit(current_X_train, y_train_final_for_model)
            train_time = datetime.now() - start_time
            print(f"  Entraînement terminé en {train_time}")

            print(f"  Évaluation {name} sur Test Set...")
            current_X_test = X_test_scaled.fillna(0) # Fallback fillna
            y_pred_test = model.predict(current_X_test)
            y_proba_test = model.predict_proba(current_X_test)[:, 1]

            print(f"\n  Classification Report (Test - {name}):")
            print(classification_report(y_test, y_pred_test, zero_division=0))
            cm_model = confusion_matrix(y_test, y_pred_test)
            print(f"  Matrice de Confusion (Test - {name}):\n{cm_model}")

            roc_auc = roc_auc_score(y_test, y_proba_test) if len(np.unique(y_test)) > 1 else 0.5
            pr_auc = average_precision_score(y_test, y_proba_test)
            report_dict = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)

            print(f"\n  AUC ROC: {roc_auc:.4f}, AUC PR: {pr_auc:.4f}")
            benchmark_results_all_models[name] = {
                 'AUC_ROC': roc_auc, 'AUC_PR': pr_auc,
                 'F1_Class1': report_dict.get('1', {}).get('f1-score', 0),
                 'Recall_Class1': report_dict.get('1', {}).get('recall', 0),
                 'Precision_Class1': report_dict.get('1', {}).get('precision', 0),
                 'Training_Time': str(train_time)
            }
        except Exception as e:
            print(f"  ERREUR pour {name}: {e}")
            benchmark_results_all_models[name] = {'AUC_ROC': np.nan, 'AUC_PR': np.nan, 'F1_Class1': np.nan, 'Training_Time': 'Error'}
else:
    print("\nErreur: Données non prêtes pour le benchmark des modèles (Cellule 4A).")
# --- FIN Cellule 4A ---



--- Cellule 4A: Benchmark des Modèles ML (Sauf SVC) ---
--- (Entraînement sur Données Déséquilibrées) ---

--- Modèle Testé : LogisticRegression ---
  Entraînement LogisticRegression...
  Entraînement terminé en 0:00:56.786703
  Évaluation LogisticRegression sur Test Set...

  Classification Report (Test - LogisticRegression):
              precision    recall  f1-score   support

           0       0.94      0.99      0.96    194271
           1       0.64      0.28      0.39     15986

    accuracy                           0.93    210257
   macro avg       0.79      0.63      0.68    210257
weighted avg       0.92      0.93      0.92    210257

  Matrice de Confusion (Test - LogisticRegression):
[[191807   2464]
 [ 11553   4433]]

  AUC ROC: 0.9041, AUC PR: 0.4985

--- Modèle Testé : RandomForest ---
  Entraînement RandomForest...
  Entraînement terminé en 0:02:47.085168
  Évaluation RandomForest sur Test Set...

  Classification Report (Test - RandomForest):
              precisio

In [5]:
if data_ready_for_modeling and len(benchmark_results_all_models) > 0:
    print("\n\n--- Cellule 5: Résultats Complets du Benchmark des Modèles (Sans Équilibrage Explicite) ---")
    results_df_benchmark_models = pd.DataFrame(benchmark_results_all_models).T
    # S'assurer que les colonnes numériques sont bien de type float pour le tri et l'affichage
    cols_to_convert = ['AUC_ROC', 'AUC_PR', 'F1_Class1', 'Recall_Class1', 'Precision_Class1']
    for col in cols_to_convert:
        if col in results_df_benchmark_models.columns:
            results_df_benchmark_models[col] = pd.to_numeric(results_df_benchmark_models[col], errors='coerce')

    results_df_benchmark_models = results_df_benchmark_models.sort_values(by='AUC_PR', ascending=False)
    pd.options.display.float_format = '{:.4f}'.format
    print(results_df_benchmark_models)
elif not data_ready_for_modeling:
    print("\nErreur: Données non prêtes, aucun résultat de benchmark à afficher.")
else:
    print("\nAucun résultat de benchmark n'a été collecté.")
# --- FIN Cellule 5 ---



--- Cellule 5: Résultats Complets du Benchmark des Modèles (Sans Équilibrage Explicite) ---
                    AUC_ROC  AUC_PR  F1_Class1  Recall_Class1  \
LightGBM             0.9288  0.5968     0.4956         0.3827   
GradientBoosting     0.9268  0.5910     0.4870         0.3723   
MLPClassifier        0.9267  0.5900     0.4751         0.3545   
RandomForest         0.9253  0.5796     0.4236         0.2963   
LogisticRegression   0.9041  0.4985     0.3874         0.2773   
KNeighbors           0.8560  0.4475     0.3946         0.2794   

                    Precision_Class1   Training_Time  
LightGBM                      0.7028  0:00:12.551792  
GradientBoosting              0.7038  0:37:06.392225  
MLPClassifier                 0.7202  0:02:37.944568  
RandomForest                  0.7426  0:02:47.085168  
LogisticRegression            0.6427  0:00:56.786703  
KNeighbors                    0.6711  0:00:01.817785  
